# Preliminary calculations

In this section, we will perform some preliminary calculations to provide a basis for our discussion with stakeholders. Here are some suggestions for basic analysis/statistics that can be valuable:

1. Km2 (or hectares) of the different rangelands systems at global level and per country. This will provide a global and country-specific overview of the rangelands systems, which can be useful in discussions about land use and management.

2. Km2 (or hectares) of annual forest loss over rangelands systems per country. This will give us an understanding of the extent of forest loss in different countries, which can be a crucial factor in environmental and conservation discussions.

3. Annual average and trend of Net primary production (NPP) per rangeland system and per country. This will give us insights into the productivity of different rangeland systems and how it's changing over time, which can be important in discussions about agricultural productivity and food security.


## Setup

### Library import


In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pyproj import CRS
from shapely.ops import unary_union
from tqdm import tqdm

### Utils

In [ ]:
def rtree_intersect(
    gdf_intersected,
    gdf_intersecting,
    intersected_column_name="rangeland",
    intersecting_columns_to_keep=None,
):
    """
    This function intersects two GeoDataFrames using an R-tree spatial index.
    Parameters
    ----------
    gdf_intersected : GeoDataFrame
        The GeoDataFrame that will be intersected.
    gdf_intersecting : GeoDataFrame

    Returns
    -------
    List of dictionaries with the area of the intersected geometries.
    """
    if intersecting_columns_to_keep is None:
        intersecting_columns_to_keep = []
    values = []
    # Create a spatial index
    sindex = gdf_intersected.sindex
    # we iterate over the countries
    for _index, row in tqdm(gdf_intersecting.iterrows()):
        geometry = row["geometry"]
        # rangelands that intersect with a country
        possible_matches_index = list(sindex.intersection(geometry.bounds))
        possible_matches = gdf_intersected.iloc[possible_matches_index]
        # intersection between the rangelands and the country
        precise_matches = possible_matches.intersection(geometry)
        # transform the precise_matches into a GeoDataFrame
        precise_matches_gdf = gpd.GeoDataFrame({"geometry": precise_matches}).reset_index()
        precise_matches_gdf = precise_matches_gdf[~precise_matches_gdf.geometry.is_empty]
        precise_matches_index = precise_matches_gdf["index"].to_list()
        precise_matches_gdf = gpd.GeoDataFrame(
            pd.merge(
                gdf_intersected.iloc[precise_matches_index]
                .reset_index()
                .drop(columns=["geometry"]),
                precise_matches_gdf,
                on="index",
                how="left",
            )
        )

        # Computing the area of different rangelands systems
        # reproject in Mollweide
        precise_matches_gdf = precise_matches_gdf.to_crs("ESRI:54009")
        # compute area in square kilometers
        precise_matches_gdf["area_km2"] = precise_matches_gdf["geometry"].area / 10**6
        precise_matches_gdf.sort_values(by="area_km2", ascending=False)

        df = precise_matches_gdf[[intersected_column_name, "area_km2"]].copy()

        if df.empty:
            df = pd.DataFrame({"rangeland": [""], "area_km2": [np.nan]})

        for column in intersecting_columns_to_keep:
            df.loc[:, column] = row[column]

        values.append(df)

    df = pd.concat(values)
    df.reset_index(drop=True, inplace=True)
    df = df[intersecting_columns_to_keep + [intersected_column_name, "area_km2"]]

    return df

## Global and Country-Specific Overview of Rangelands Systems

We will start by analyzing the extent of different rangelands systems at the global level and per country. This will provide us with a basic understanding of the distribution of rangelands systems and their importance in different regions.

### Read data


**Countries**

In [ ]:
countries = gpd.read_parquet("../data/processed/countries.parquet")

**Global terrrestrial surface**

In [ ]:
geometry = unary_union(countries["geometry"])

earth = gpd.GeoDataFrame(
    {"name": ["Global Terrestrial Surface"], "geometry": [geometry]}, crs=CRS("EPSG:4326")
)

earth

**Rangeland ecoregions**

In [ ]:
ecoregions = gpd.read_file("../data/processed/ecoregions_2017.geojson")
ecoregions.columns = ecoregions.columns.str.lower()
ecoregions.head()

**Rangelad biomes**

In [ ]:
rangelands_list = list(ecoregions["biome_name"].unique())
geometries = [
    unary_union(ecoregions[ecoregions["biome_name"] == rangeland]["geometry"])
    for rangeland in rangelands_list
]

biomes = gpd.GeoDataFrame(
    {"biome_name": rangelands_list, "geometry": geometries}, crs=CRS("EPSG:4326")
)

biomes

**Global rangeland system**

In [ ]:
geometry = unary_union(biomes["geometry"])

rangeland = gpd.GeoDataFrame(
    {"name": ["Global Rangeland System"], "geometry": [geometry]}, crs=CRS("EPSG:4326")
)

rangeland

**Display data**

In [ ]:
# Define a color for each unique rangeland
color_dict = {
    "Tundra": "#b5c58f",
    "Mediterranean Forests, Woodlands & Scrub": "#ccb879",
    "Deserts & Xeric Shrublands": "#dfd9c2",
    "Temperate Grasslands, Savannas & Shrublands": "#dcd939",
    "Montane Grasslands & Shrublands": "#ab6c28",
    "Flooded Grasslands & Savannas": "#b8d9eb",
    "Tropical & Subtropical Grasslands, Savannas & Shrublands": "#6c9fb8",
}

fig, ax = plt.subplots(1, 1, figsize=(10, 10))
ax.set_facecolor("darkblue")

# Plot the countries
countries.plot(ax=ax, color="white", edgecolor="black")

# Plot each biome with its corresponding color
for i in range(len(biomes) - 1):
    data = gpd.GeoDataFrame(biomes.iloc[i : i + 1])
    data.plot(ax=ax, color=color_dict[data["biome_name"].iloc[0]], alpha=0.75)

plt.title("Countries of the World")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()

### Compute statistics

**Global rangeland system**
- km² of the global rangeland system
- % of the global rangeland system from total Earth area

In [ ]:
df_earth = earth.to_crs("ESRI:54009")
# Compute area in square kilometers
earth["area_km2"] = df_earth["geometry"].area / 10**6

# Reorder the columns
cols = list(earth.columns)
cols.remove("geometry")
cols.append("geometry")
earth = earth[cols]


earth

In [ ]:
df_rangeland = rangeland.to_crs("ESRI:54009")
# Compute area in square kilometers
rangeland["area_km2"] = df_rangeland["geometry"].area / 10**6

# Compute percentage from total earth area
rangeland["percentage"] = rangeland["area_km2"] / earth["area_km2"].iloc[0] * 100

# Reorder the columns
cols = list(rangeland.columns)
cols.remove("geometry")
cols.append("geometry")
rangeland = rangeland[cols]

rangeland

**Rangeland biomes**
- km² of the different rangelands biomes at global level
- % of the different rangelands biomes from total rangelands area

In [ ]:
df_biomes = biomes.copy()
# Reproject in Mollweide
df_biomes = df_biomes.to_crs("ESRI:54009")
# Compute area in square kilometers
biomes["area_km2"] = df_biomes["geometry"].area / 10**6

# Compute percentage from total Global Rangeland System area
biomes["percentage"] = biomes["area_km2"] / rangeland["area_km2"].iloc[0] * 100

# Reorder the columns
cols = list(biomes.columns)
cols.remove("geometry")
cols.append("geometry")
biomes = biomes[cols]

biomes

**Rangeland ecoregions**
- km² of the different rangelands ecoregions at global level
- % of the different rangelands ecoregions from total rangeland biomes area

In [ ]:
biome_list = list(ecoregions["biome_name"].unique())

for biome in biome_list:
    df_ecoregions = ecoregions[ecoregions["biome_name"] == biome].copy()

    # Reproject in Mollweide
    df_ecoregions = df_ecoregions.to_crs("ESRI:54009")
    # Compute area in square kilometers
    ecoregions.loc[ecoregions["biome_name"] == biome, "area_km2"] = (
        df_ecoregions["geometry"].area / 10**6
    )

    # Compute percentage from total Rangeland Biome area
    ecoregions.loc[ecoregions["biome_name"] == biome, "percentage"] = (
        ecoregions.loc[ecoregions["biome_name"] == biome, "area_km2"]
        / biomes[biomes["biome_name"] == biome]["area_km2"].iloc[0]
        * 100
    )


# Reorder the columns
cols = list(ecoregions.columns)
cols.remove("geometry")
cols.append("geometry")
ecoregions = ecoregions[cols]

ecoregions

### Save data

In [ ]:
ecoregions.to_file(("../data/processed/rageland_ecoregions.geojson"), driver="GeoJSON")
biomes.to_file(("../data/processed/rageland_biomes.geojson"), driver="GeoJSON")
biomes.to_file(("../data/processed/rageland_biomes.geojson"), driver="GeoJSON")
rangeland.to_file(("../data/processed/rageland_system.geojson"), driver="GeoJSON")

*** 
## Old calulations

### Compute statistics of different rangelands systems at the global level
- km² (or hectares) of total sum of rangeland systems at global level
- km² (or hectares) of the different rangelands systems at global level
- % of the different rangelands systems from total rangelands area
- % of the different rangelands systems from total Earth area


In [ ]:
df_earth = earth.to_crs("ESRI:54009")
# Compute area in square kilometers
df_earth["area_km2"] = df_earth["geometry"].area / 10**6
df_earth

In [ ]:
rangelands = rangeland.copy()

In [ ]:
df_global = rangelands.copy()
# Reproject in Mollweide
df_global = df_global.to_crs("ESRI:54009")
# Compute area in square kilometers
df_global["area_km2"] = df_global["geometry"].area / 10**6
# Drop the geometry column
df_global.drop(columns="geometry", inplace=True)
# Sort by area
df_global.sort_values(by="area_km2", ascending=False, inplace=True)
# Compute percentage of total area
df_global["percentage_rangelands_area"] = (
    df_global["area_km2"] / df_global["area_km2"].iloc[0] * 100
)
df_global["percentage_earth_area"] = df_global["area_km2"] / df_earth["area_km2"].iloc[0] * 100
df_global

**Save data**

In [ ]:
df_global.to_csv("../data/processed/rangelands_global.csv", index=False)
df_global.to_excel("../data/processed/rangelands_global.xlsx", index=False)

### Compute the area of different rangelands systems by country
#### First compute the area of different rangelands systems for a single country
**Intersecting the rangelands systems with a country boundary**

In [ ]:
# Get the geometry of Spain
country = countries[countries["ISO3_CODE"] == "ESP"]
geometry = country["geometry"].iloc[0]
geometry

In [ ]:
# Intersect the rangelands with Spain
sindex = rangelands.sindex
# ecoregions that intersect with a country
possible_matches_index = list(sindex.intersection(geometry.bounds))
possible_matches = rangelands.iloc[possible_matches_index]
# intersection between the rangelands and the country
precise_matches = possible_matches.intersection(geometry)
# transform the precise_matches into a GeoDataFrame
precise_matches_gdf = gpd.GeoDataFrame({"geometry": precise_matches}).reset_index()
precise_matches_gdf = precise_matches_gdf[~precise_matches_gdf.geometry.is_empty]
precise_matches_index = precise_matches_gdf["index"].to_list()
precise_matches_gdf = gpd.GeoDataFrame(
    pd.merge(
        rangelands.iloc[precise_matches_index].reset_index().drop(columns=["geometry"]),
        precise_matches_gdf,
        on="index",
        how="left",
    )
)

**Computing the area of different rangelands systems for a single country**

In [ ]:
df = precise_matches_gdf.copy()
# Reproject in Mollweide
df = df.to_crs("ESRI:54009")
# Compute area in square kilometers
df["area_km2"] = df["geometry"].area / 10**6
# Drop the geometry column
df.drop(columns=["index", "geometry"], inplace=True)
# Sort by area
df.sort_values(by="area_km2", ascending=False, inplace=True)
df

**Display the rangelands systems of a specific country**

In [ ]:
fig, ax = plt.subplots(figsize=[5, 5])
ax.set_aspect("equal")

possible_matches.plot(ax=ax, color="r")
precise_matches_gdf.plot(ax=ax, color="g")
country.plot(ax=ax, color="w", edgecolor="b", alpha=0.5)

plt.xlim(geometry.bounds[0] - 0.1, geometry.bounds[2] + 0.1)
plt.ylim(geometry.bounds[1] - 0.1, geometry.bounds[3] + 0.1)

#### Compute the area of different rangelands systems by country

In [ ]:
df_countries = rtree_intersect(
    rangelands,  # [~rangelands["rangeland"].isin(["All"])],
    countries,
    intersected_column_name="rangeland",
    intersecting_columns_to_keep=["CNTR_NAME", "NAME_ENGL", "ISO3_CODE"],
)
df_countries.dropna(inplace=True)
df_countries

**Compute percentages**

In [ ]:
def percentage(x, df_global):
    """
    This function computes the percentage of the area of each rangeland system.
    """
    global_area = df_global[df_global["rangeland"] == x["rangeland"]]["area_km2"].iloc[0]
    return x["area_km2"] / global_area * 100


df_countries["percentage"] = df_countries.apply(lambda x: percentage(x, df_global), axis=1)

In [ ]:
df_countries

**Save data**

In [ ]:
df_countries.to_csv("../data/processed/rangelands_countries.csv", index=False)
df_countries.to_excel("../data/processed/rangelands_countries.xlsx", index=False)